In [1]:
import os
# os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"

import sys

sys.path.insert(0, "../..")

In [2]:
import chex
import jax
import jax.numpy as jnp
import jax.random as jrandom
import math
import matplotlib.pyplot as plt
import numpy as np
import optax

from flax import nnx
from functools import partial
from jax_tqdm import loop_tqdm
from tqdm.notebook import tqdm
from typing import Any, NamedTuple, Callable, Dict

from src.models.gpt import GPT
from src.models import TrainState

In [3]:
# loss_type = "reinforce"
loss_type = "ppo"
config = {
    "clip_param": 0.2,
    "num_ppo_updates": 3,
    "minibatch_size": 32,
}
batch_size = 32

tm_alphabet_size = 3
num_registers = 0
blank_symb = 2

log_interval = 1000
lr = 1e-4
weight_decay = 0.0
max_grad_norm = False

num_states = 1
num_blocks = 2
num_heads = 2
embed_dim = 16
widening_factor = 4
dtype = jnp.float32

seed = 42
rngs = nnx.Rngs(seed)

### Model

In [4]:
class GPTTM(nnx.Module):
    """
    A GPT representing a Turing machine.

    Sequence format with N registers:
    Input:  <READ> , <R_1>, <R_2>, ..., <R_N>, <STATE>, <BLANK>
    Output: <WRITE>, <R_1>, <R_2>, ..., <R_N>, <STATE>, <DIRECTION>

    Env:
    Bounded single-tape with inputs on it.
    """

    def __init__(
        self,
        tm_alphabet_size: int,
        num_states: int,
        num_blocks: int,
        num_heads: int,
        embed_dim: int,
        widening_factor: int,
        rngs: nnx.Rngs,
        dtype = None,
        attention_fn=nnx.dot_product_attention,
        **kwargs,
    ) -> None:
        self.tm_alphabet_size = tm_alphabet_size

        self.tm_alphabet_emb = nnx.Embed(
            tm_alphabet_size,
            embed_dim,
            rngs=rngs,
            dtype=dtype,
        )

        self.state_emb = nnx.Embed(
            num_states,
            embed_dim,
            rngs=rngs,
            dtype=dtype,
        )

        self.blank_symb = nnx.Embed(
            1,
            embed_dim,
            rngs=rngs,
            dtype=dtype,
        )

        self.direction_emb = nnx.Param(
            jrandom.uniform(
                rngs.params(),
                (embed_dim, 2),
            )
        )

        self.gpt = GPT(
            num_blocks=num_blocks,
            num_heads=num_heads,
            embed_dim=embed_dim,
            widening_factor=widening_factor,
            rngs=rngs,
            use_causal_mask=False,
            attention_fn=attention_fn,
            dtype=dtype,
        )
        
        self.num_heads = num_heads
        self.embed_dim = embed_dim

    def __call__(
        self,
        batch: Any,
    ):
        batch_size = len(batch["sequence"])
        # <READ>, <R_1>, ..., <R_N>
        sequences = self.tm_alphabet_emb(batch["sequence"])
        # <STATE>
        states = self.state_emb(batch["state"])
        # <BLANK>
        blanks = self.blank_symb(
            np.zeros((batch_size, 1), dtype=int),
        )

        # Process input
        token_seq = self.gpt(jnp.concatenate((sequences, states, blanks), axis=1))

        # <WRITE>, <R_1>, ..., <R_N>
        sequences = token_seq[:, :-2] @ jax.lax.stop_gradient(
            self.tm_alphabet_emb.embedding.T
        )
        # <STATE>
        states = token_seq[:, -2:-1] @ jax.lax.stop_gradient(
            self.state_emb.embedding.T
        )
        # <DIRECTION>
        directions = token_seq[:, -1:] @ self.direction_emb

        return (
            sequences,
            states,
            directions,
        )


### Environment

In [18]:
def make_generation_batch(num_questions, max_input_len, max_tm_tape_len):
    @jax.jit
    def generation_batch(rng):
        question_len = jrandom.randint(
            rng,
            shape=(num_questions,),
            minval=1,
            maxval=max_input_len,
        )
        solution_len = question_len

        question_mask = 1 - jnp.cumsum(jnp.eye(max_tm_tape_len)[question_len], axis=-1)
        question_bits = jrandom.bernoulli(rng, shape=question_mask.shape)
        all_blanks = jnp.full(
            (num_questions, max_tm_tape_len),
            fill_value=2,
            dtype=int,
        )

        question = question_bits * question_mask + all_blanks * (1 - question_mask)
        question = question.astype(int)

        solution = jnp.copy(question)

        return {
            "question": question,
            "solution": solution,
            "question_len": question_len,
            "solution_len": solution_len,
        }
    return generation_batch

def make_eval_generation_batch(num_questions, max_input_len, max_tm_tape_len):
    @jax.jit
    def generation_batch(rng):
        question_len = jnp.full(num_questions, fill_value=max_input_len - 1, dtype=int)
        solution_len = question_len

        question_mask = 1 - jnp.cumsum(jnp.eye(max_tm_tape_len)[question_len], axis=-1)
        question_bits = jrandom.bernoulli(rng, shape=question_mask.shape)
        all_blanks = jnp.full(
            (num_questions, max_tm_tape_len),
            fill_value=2,
            dtype=int,
        )

        question = question_bits * question_mask + all_blanks * (1 - question_mask)
        question = question.astype(int)

        solution = jnp.copy(question)

        return {
            "question": question,
            "solution": solution,
            "question_len": question_len,
            "solution_len": solution_len,
        }
    return generation_batch

In [6]:
class StepState(NamedTuple):
    graphdef: Any
    params: Any
    rest: Any
    rng: chex.PRNGKey
    questions: chex.Array
    read_pointer: chex.Array
    observations: chex.Array
    actions: chex.Array
    output_tape: chex.Array
    output_pointer: chex.Array
    blank_symb: int
    step_i: int = 0
    deterministic: int = 0

class RolloutResult(NamedTuple):
    observations: chex.Array
    actions: chex.Array
    output_tape: chex.Array
    success: chex.Array

In [7]:
def predict_step(
    step_state: StepState,
):
    step_i = step_state.step_i
    rng, rng_step = jrandom.split(step_state.rng, 2)

    batch_size, max_question_len = step_state.questions.shape

    module = nnx.merge(step_state.graphdef, step_state.params, step_state.rest)
    module.eval()
    module.set_attributes(deterministic=True)
    logits = module({
        "sequence": step_state.observations[:, step_i, :-2],
        "state": step_state.observations[:, step_i, -2:-1],
    })

    next_sequence = jax.lax.cond(
        step_state.deterministic,
        lambda rng_step, logits: jnp.argmax(logits, axis=-1),
        jrandom.categorical,
        rng_step,
        logits[0],
    )

    next_state = jax.lax.cond(
        step_state.deterministic,
        lambda rng_step, logits: jnp.argmax(logits, axis=-1),
        jrandom.categorical,
        rng_step,
        logits[1],
    )

    direction = jax.lax.cond(
        step_state.deterministic,
        lambda rng_step, logits: jnp.argmax(logits, axis=-1),
        jrandom.categorical,
        rng_step,
        logits[2],
    )

    # Update output pointer and output tape
    output_tape = step_state.output_tape.at[
        jnp.arange(batch_size), step_state.output_pointer
    ].set(next_sequence[:, 0])
    output_pointer = jnp.where(
        next_sequence[:, 0] == step_state.blank_symb,
        step_state.output_pointer,
        step_state.output_pointer + 1,
    )

    # Update read pointer
    next_read_pointer = step_state.read_pointer + ((-1) ** direction[..., 0]).astype(int)
    # jax.debug.print("{x}", x=next_read_pointer)
    read_pointer = jnp.where(
        jnp.logical_or(
            next_read_pointer < 0,
            next_read_pointer >= max_question_len,
        ),
        step_state.read_pointer,
        next_read_pointer,
    )

    # Update observation/action pairs
    observations = step_state.observations.at[:, step_i + 1, 0].set(
        step_state.questions[jnp.arange(batch_size), read_pointer]
    )
    observations = observations.at[:, step_i + 1, -2:-1].set(
        next_state
    )

    actions = step_state.actions.at[:, step_i + 1].set(
        jnp.concatenate(
            (next_sequence, next_state, direction),
            axis=1,
        )
    )

    # jax.debug.print("{x}, {y}, {z}", x=next_sequence, y=next_state, z=direction)

    return StepState(
        graphdef=step_state.graphdef,
        params=step_state.params,
        rest=step_state.rest,
        rng=rng,
        questions=step_state.questions,
        read_pointer=read_pointer,
        observations=observations,
        actions=actions,
        output_tape=output_tape,
        output_pointer=output_pointer,
        blank_symb=step_state.blank_symb,
        step_i=step_i + 1,
        deterministic=step_state.deterministic,
    )


def make_rollout(graphdef, rest):
    @partial(nnx.jit, static_argnames=["num_registers"])
    def rollout(
        params: Any,
        rng: chex.PRNGKey,
        batch: Any,
        num_registers: int,
        blank_symb: int,
        deterministic: int = 0,
    ):  
        question = batch["question"]
        solution = batch["solution"]
        num_questions, max_step = question.shape

        observations = jnp.full(
            (num_questions, max_step, 3 + num_registers),
            fill_value=blank_symb,
            dtype=int,
        )
        actions = jnp.full_like(observations, fill_value=blank_symb, dtype=int)
        output_tape = jnp.full_like(solution, fill_value=blank_symb, dtype=int)

        # First <READ> is first token
        observations = observations.at[:, 0, 0].set(question[..., 0])
        observations = observations.at[:, 0, -2:].set(0)
        observations = observations.at[:, :, -1:].set(0)

        step_state = StepState(
            graphdef=graphdef,
            params=params,
            rest=rest,
            rng=rng,
            questions=question,
            read_pointer=jnp.zeros(num_questions, dtype=int),
            observations=observations,
            actions=actions,
            output_tape=output_tape,
            output_pointer=jnp.zeros(num_questions, dtype=int),
            blank_symb=blank_symb,
            deterministic=deterministic,
            step_i=0,
        )

        step_state = jax.lax.while_loop(
            lambda state: state.step_i < max_step - 1,
            predict_step,
            step_state,
        )

        success = jnp.all(step_state.output_tape == solution, axis=-1)

        return RolloutResult(
            observations=step_state.observations[:, :-1],
            actions=step_state.actions[:, 1:],
            output_tape=step_state.output_tape,
            success=success,
        )
    return rollout

### Learner

#### REINFORCE

In [8]:
def make_reinforce_loss(graphdef, rest, config):
    def reinforce_loss(
        params: nnx.Param,
        rollout_res: RolloutResult,
        batch: Dict[str, Any],
    ):
        model = nnx.merge(graphdef, params, rest)
        model.set_attributes(deterministic=False, decode=False)
        batch_size, rollout_len = rollout_res.observations.shape[:2]
        observations = rollout_res.observations.reshape((batch_size * rollout_len), -1)
        actions = rollout_res.actions.reshape((batch_size * rollout_len), -1)
        logits = model({
            "sequence": observations[:, :-2],
            "state": observations[:, -2:-1],
        })

        sequence_actions = jax.nn.one_hot(actions[:, :-2], num_classes=logits[0].shape[-1])
        state_actions = jax.nn.one_hot(actions[:, -2:-1], num_classes=logits[1].shape[-1])
        direction_actions = jax.nn.one_hot(actions[:, -1:], num_classes=logits[2].shape[-1])

        sequence_lprobs = jnp.sum(
            logits[0], axis=-1, where=sequence_actions,
        ) - jax.nn.logsumexp(logits[0], axis=-1)
        state_lprobs = jnp.sum(
            logits[1], axis=-1, where=state_actions,
        ) - jax.nn.logsumexp(logits[1], axis=-1)
        direction_lprobs = jnp.sum(
            logits[2], axis=-1, where=direction_actions,
        ) - jax.nn.logsumexp(logits[2], axis=-1)

        total_lprobs = (
            jnp.sum(sequence_lprobs, axis=-1)
            + jnp.sum(state_lprobs, axis=-1)
            + jnp.sum(direction_lprobs, axis=-1)
        )

        sequence_probs = jax.nn.softmax(logits[0], axis=-1)
        sequence_entropy = jnp.mean(optax.softmax_cross_entropy(logits[0], sequence_probs))

        state_probs = jax.nn.softmax(logits[1], axis=-1)
        state_entropy = jnp.mean(optax.softmax_cross_entropy(logits[1], state_probs))

        direction_probs = jax.nn.softmax(logits[2], axis=-1)
        direction_entropy = jnp.mean(optax.softmax_cross_entropy(logits[2], direction_probs))

        return -jnp.mean(
            total_lprobs
            * jnp.repeat(rollout_res.success, repeats=rollout_len, axis=0)
            # * jnp.repeat((-1) ** (1 - rollout_res.success), repeats=rollout_len, axis=0)
            # * jnp.repeat(jnp.sum(batch["solution"] == rollout_res.output_tape, axis=-1), repeats=rollout_len, axis=0)
        ), {
            "sequence_entropy": sequence_entropy,
            "state_entropy": state_entropy,
            "direction_entropy": direction_entropy,
        }
    
    compute_loss_and_grad = jax.jit(
        jax.value_and_grad(
            reinforce_loss,
            has_aux=True,
        ),
    )

    def reinforce_update_rule(train_state, rollout_res, batch):
        (loss, aux), grads = compute_loss_and_grad(
            train_state.params,
            rollout_res,
            batch,
        )
        train_state = train_state.apply_gradients(grads=grads)
        return train_state, loss, aux

    return reinforce_update_rule


#### PPO

In [9]:
def make_ppo_loss(graphdef, rest, config):
    def _lprobs(params, rollout_res):
        model = nnx.merge(graphdef, params, rest)
        model.set_attributes(deterministic=False, decode=False)
        batch_size, rollout_len = rollout_res.observations.shape[:2]
        observations = rollout_res.observations.reshape((batch_size * rollout_len), -1)
        actions = rollout_res.actions.reshape((batch_size * rollout_len), -1)
        logits = model({
            "sequence": observations[:, :-2],
            "state": observations[:, -2:-1],
        })

        sequence_actions = jax.nn.one_hot(actions[:, :-2], num_classes=logits[0].shape[-1])
        state_actions = jax.nn.one_hot(actions[:, -2:-1], num_classes=logits[1].shape[-1])
        direction_actions = jax.nn.one_hot(actions[:, -1:], num_classes=logits[2].shape[-1])

        sequence_lprobs = jnp.sum(
            logits[0], axis=-1, where=sequence_actions,
        ) - jax.nn.logsumexp(logits[0], axis=-1)
        state_lprobs = jnp.sum(
            logits[1], axis=-1, where=state_actions,
        ) - jax.nn.logsumexp(logits[1], axis=-1)
        direction_lprobs = jnp.sum(
            logits[2], axis=-1, where=direction_actions,
        ) - jax.nn.logsumexp(logits[2], axis=-1)

        total_lprobs = (
            jnp.sum(sequence_lprobs, axis=-1)
            + jnp.sum(state_lprobs, axis=-1)
            + jnp.sum(direction_lprobs, axis=-1)
        )
        return total_lprobs, logits

    def ppo_loss(
        params: nnx.Param,
        rollout_res: RolloutResult,
        batch: Dict[str, Any],
    ):
        total_lprobs, logits = _lprobs(
            params,
            rollout_res,
        )

        returns = jnp.repeat(rollout_res.success, repeats=rollout_res.observations.shape[1], axis=0)

        is_ratio = jnp.exp(total_lprobs - batch["old_lprobs"])
        # XXX: Deal with inf values
        is_ratio = jax.lax.select(
            jnp.isfinite(is_ratio), is_ratio, jnp.zeros_like(is_ratio)
        )

        clipped_is_ratio = jnp.clip(
            is_ratio,
            a_min=1 - config["clip_param"],
            a_max=1 + config["clip_param"],
        )

        surrogate_1 = is_ratio * returns
        surrogate_2 = clipped_is_ratio * returns
        pi_surrogate = jnp.minimum(surrogate_1, surrogate_2)

        sequence_probs = jax.nn.softmax(logits[0], axis=-1)
        sequence_entropy = jnp.mean(optax.softmax_cross_entropy(logits[0], sequence_probs))

        state_probs = jax.nn.softmax(logits[1], axis=-1)
        state_entropy = jnp.mean(optax.softmax_cross_entropy(logits[1], state_probs))

        direction_probs = jax.nn.softmax(logits[2], axis=-1)
        direction_entropy = jnp.mean(optax.softmax_cross_entropy(logits[2], direction_probs))

        pi_surrogate = jrandom.permutation(batch["rng"], pi_surrogate)[:config["minibatch_size"]]
        return -jnp.mean(pi_surrogate), {
            "sequence_entropy": sequence_entropy,
            "state_entropy": state_entropy,
            "direction_entropy": direction_entropy,
        }
    
    compute_loss_and_grad = jax.jit(
        jax.value_and_grad(
            ppo_loss,
            has_aux=True,
        ),
    )

    class PPOTrainState(NamedTuple):
        train_state: TrainState
        loss: float
        aux: Dict

    def ppo_update_rule(train_state, rollout_res, batch):
        old_lprobs, _ = _lprobs(train_state.params, rollout_res)
        batch["old_lprobs"] = old_lprobs

        def _update_params(iter_i, state):
            batch["rng"] = jrandom.fold_in(batch["rng"], iter_i)
            (loss, aux), grads = compute_loss_and_grad(
                state.train_state.params,
                rollout_res,
                batch,
            )
            train_state = state.train_state.apply_gradients(grads=grads)
            return PPOTrainState(train_state, loss, aux)

        state = jax.lax.fori_loop(
            0,
            config["num_ppo_updates"],
            _update_params,
            PPOTrainState(
                train_state,
                0.0,
                {
                    "sequence_entropy": 0.0,
                    "state_entropy": 0.0,
                    "direction_entropy": 0.0,
                }
            )
        )
        return state.train_state, state.loss, state.aux

    return ppo_update_rule

### Instantiation

In [10]:
model = GPTTM(
    tm_alphabet_size,
    num_states,
    num_blocks,
    num_heads,
    embed_dim,
    widening_factor,
    rngs,
    dtype,
)

graphdef, params, rest = nnx.split(model, nnx.Param, ...)

opt_transforms = []
if max_grad_norm:
    opt_transforms.append(
        optax.clip_by_global_norm(max_grad_norm)
    )
opt_transforms.append(optax.adamw(lr, weight_decay=weight_decay))
opt = optax.chain(*opt_transforms)

train_state = TrainState.create(
    apply_fn=graphdef.apply,
    params=params,
    tx=opt,
    graphdef=graphdef,
    rest=rest,
)

update_rule = (make_reinforce_loss if loss_type == "reinforce" else make_ppo_loss)(graphdef, rest, config)
rollout = make_rollout(graphdef, rest)

### Trainer

In [11]:
def l2_norm(params):
    return sum(jnp.sum(p**2) for p in jax.tree_util.tree_leaves(params))

class LogStepState(NamedTuple):
    train_state: TrainState
    losses: chex.Array
    successes: chex.Array

def make_train_step(
    update_rule,
    batch_size,
    max_input_len,
    max_tm_tape_len,
    max_iter,
    print_rate,
    sample_rng,
    rollout_rng,
):
    sample_batch = make_generation_batch(
        batch_size,
        max_input_len,
        max_tm_tape_len,
    )

    @jax.jit
    @loop_tqdm(max_iter, print_rate=print_rate)
    def train_step(
        step_i: int,
        step_state: LogStepState,
    ):
        train_state = step_state.train_state
        rng = jrandom.fold_in(sample_rng, step_i)

        batch = sample_batch(rng)

        rollout_res = rollout(
            train_state.params,
            rollout_rng,
            batch,
            num_registers,
            blank_symb,
        )
        batch["rng"] = rng

        train_state, loss, aux = update_rule(
            train_state,
            rollout_res,
            batch,
        )

        success_rate = jnp.mean(rollout_res.success)
        # jax.debug.print(
        #     "Iter: {step_i} Success rate: {x}, {grad_norm} {sequence_entropy} {state_entropy} {direction_entropy}",
        #     step_i=step_i,
        #     x=success_rate,
        #     grad_norm=l2_norm(grads),
        #     **aux,
        # )

        is_log = step_i % log_interval == 0
        log_idx = step_i // log_interval
        losses = jax.lax.select(
            is_log,
            step_state.losses.at[log_idx].set(loss),
            step_state.losses,
        )
        successes = jax.lax.select(
            is_log,
            step_state.successes.at[log_idx].set(success_rate),
            step_state.successes,
        )

        return LogStepState(
            train_state,
            losses,
            successes,
        )
    return train_step

### Experiments

In [12]:
max_iter = 1000
# max_input_len = 2
# max_tm_tape_len = 8

for max_input_len in range(2, 10):
    max_tm_tape_len = max_input_len
    logs = jax.lax.fori_loop(
        0,
        max_iter,
        make_train_step(
            update_rule,
            batch_size,
            max_input_len,
            max_tm_tape_len,
            max_iter,
            100,
            jrandom.PRNGKey(max_input_len + seed),
            jrandom.PRNGKey(max_input_len + seed + 1),
        ),
        LogStepState(
            train_state,
            jnp.zeros(math.ceil(max_iter / log_interval)),
            jnp.zeros(math.ceil(max_iter / log_interval)),
        )
    )
    train_state = logs.train_state
    print(logs.successes)

Running for 1,000 iterations:   0%|          | 0/1000 [00:00<?, ?it/s]

[0.5625]


Running for 1,000 iterations:   0%|          | 0/1000 [00:00<?, ?it/s]

[0.625]


Running for 1,000 iterations:   0%|          | 0/1000 [00:00<?, ?it/s]

...

[1.]


Running for 1,000 iterations:   0%|          | 0/1000 [00:00<?, ?it/s]

[1.]


Running for 1,000 iterations:   0%|          | 0/1000 [00:00<?, ?it/s]

[1.]


In [42]:
rng = jrandom.PRNGKey(1000)
rollout_rng = jrandom.PRNGKey(1000)
sample_batch = make_eval_generation_batch(
    batch_size,
    max_input_len + 2000,
    max_tm_tape_len + 2000,
)
batch = sample_batch(rng)

In [43]:
batch["question"]

Array([[1, 0, 0, ..., 1, 1, 2],
       [1, 1, 1, ..., 0, 0, 2],
       [1, 1, 0, ..., 1, 1, 2],
       ...,
       [1, 1, 0, ..., 0, 1, 2],
       [1, 0, 0, ..., 0, 1, 2],
       [1, 0, 1, ..., 1, 0, 2]], dtype=int32)

In [44]:
rollout_res = rollout(
    train_state.params,
    rollout_rng,
    batch,
    num_registers,
    blank_symb,
)

In [45]:
rollout_res.success

Array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True, False,
        True,  True,  True,  True,  True], dtype=bool)